Clase 011 · `011_matplotlib2` · Construye un *año típico* a partir de seis años de datos:
promedia cada hora, agrupa cada hora del calendario a lo largo de los años y guarda el
resultado en `data/003_results/`. Las dos libretas siguientes —el heatmap y las barras de
confort— parten de este archivo.

::: {.callout-important title="Datos"}
Lee `ClimaLab_2020-08-05_2026-08-05.parquet` de `clases/011_matplotlib2/data/001_raw/` (se
comparte en clase) y escribe `anio_promedio-2020-2026.parquet` en `data/003_results/`.
:::

## Librerías

`pandas` y `matplotlib`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## Promedios horarios

`resample("h").mean()` pasa de un registro por minuto a uno por hora: 52 603 filas, que
son los seis años de agosto de 2020 a agosto de 2026. Las irradiancias valen cero de noche
y `p_atm` y `rain_acc` conservan sus huecos como `NaN`.

In [2]:
f = "../data/001_raw/ClimaLab_2020-08-05_2026-08-05.parquet"
esolmet = pd.read_parquet(f)
esolmet_h = esolmet.resample("h").mean()
esolmet_h

variable,dhi,dni,ghi,p_atm,rain_acc,rh,tdb,uv,wd,ws
date,,,,,,,,,,
2020-08-05 00:00:00,0.000000,0.000000,0.000000,879.389517,1.9,95.533333,19.861667,0.000000,NaN,1.384667
2020-08-05 01:00:00,0.000000,0.000000,0.000000,878.581350,2.6,98.366667,19.628333,0.000000,NaN,0.272167
2020-08-05 02:00:00,0.000000,0.000000,0.000000,877.807200,1.5,99.316667,19.488333,0.000000,NaN,0.119667
2020-08-05 03:00:00,0.000000,0.000000,0.000000,877.222583,6.3,98.966667,19.496667,0.000000,NaN,0.678667
2020-08-05 04:00:00,0.000000,0.000000,0.000000,877.298800,0.2,98.616667,18.918333,0.000000,NaN,0.310667
...,...,...,...,...,...,...,...,...,...,...
2026-08-05 14:00:00,143.051667,789.648333,950.050000,862.822372,0.0,39.900667,31.088667,0.157783,162.277983,2.139533
2026-08-05 15:00:00,184.996667,381.619333,572.038333,861.456998,0.0,36.776167,31.611000,0.083183,160.288000,1.725567
2026-08-05 16:00:00,202.130000,492.186833,587.088333,860.324998,0.0,34.428500,32.827833,0.051867,150.622417,1.049950


## Comprobar que el índice quedó en horas enteras

Todos los minutos del índice son cero: cada fila es una hora en punto.

In [3]:
esolmet_h.index.minute

Index([0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       ...
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype='int32', name='date', length=52603)

## Agrupar por mes, día y hora

`groupby` acepta una lista de llaves; aquí son tres atributos del índice: mes, día y hora.
El promedio junta, por ejemplo, todas las 15 h de cada 1 de enero de los seis años. Salen
8784 filas = 366 días × 24 horas (el 29 de febrero existe porque 2024 fue bisiesto). El
resultado tiene un índice de tres niveles, incómodo para graficar: lo rehacemos abajo.

In [4]:
grupo = [esolmet_h.index.month,esolmet_h.index.day,esolmet_h.index.hour]
esolmet_h.groupby(by=grupo).mean()

variable        dhi  dni  ghi       p_atm  rain_acc         rh        tdb  \
date date date                                                              
1    1    0     0.0  0.0  0.0  873.931783  0.000000  53.409167  17.836633   
          1     0.0  0.0  0.0  873.847656  0.006667  56.673389  16.722500   
          2     0.0  0.0  0.0  873.520090  0.000000  58.403583  16.212367   
          3     0.0  0.0  0.0  873.576894  0.000000  60.288917  15.727033   
          4     0.0  0.0  0.0  873.767469  0.000000  62.737944  15.100200   
...             ...  ...  ...         ...       ...        ...        ...   
12   31   19    0.0  0.0  0.0  862.110272  0.000333  44.320222  21.350900   
          20    0.0  0.0  0.0  862.007002  0.000000  48.934444  20.198333   
          21    0.0  0.0  0.0  862.532007  1.537333  49.993694  19.806367   
          22    0.0  0.0  0.0  860.360770  0.502667  50.749028  19.289267   
          23    0.0  0.0  0.0  873.848805  0.000000  52.060556  18.466733   

variable              uv          wd        ws  
date date date                                  
1    1    0     0.000000  295.680175  1.413860  
          1     0.000000  313.849133  1.788290  
          2     0.000004  305.239542  1.901670  
          3     0.000004  309.496996  1.816437  
          4     0.000013  274.154650  1.948827  
...                  ...         ...       ...  
12   31   19    0.055713  209.548804  2.418727  
          20    0.062567  181.244776  1.550003  
          21    0.059379  262.768659  1.402153  
          22    0.003012  299.731850  1.723550  
          23    0.000000  281.520742  1.543513  

[8784 rows x 10 columns]

## La misma llave como texto con `strftime`

`strftime("%m-%d %H")` convierte cada fecha en una cadena `mes-día hora` que ignora el
año. Dos registros de años distintos con la misma hora del calendario tienen la misma
cadena, y eso es exactamente la llave que necesitamos.

In [5]:
esolmet_h.index.strftime("%m-%d %H")

Index(['08-05 00', '08-05 01', '08-05 02', '08-05 03', '08-05 04', '08-05 05',
       '08-05 06', '08-05 07', '08-05 08', '08-05 09',
       ...
       '08-05 09', '08-05 10', '08-05 11', '08-05 12', '08-05 13', '08-05 14',
       '08-05 15', '08-05 16', '08-05 17', '08-05 18'],
      dtype='str', name='date', length=52603)

## El año promedio con índice de fechas

Agrupamos por esa cadena y promediamos. Para volver a tener fechas, anteponemos `"2024-"`
a cada llave —2024 porque es bisiesto y así cabe el 29 de febrero— y `pd.to_datetime` con
`format` las convierte en un `DatetimeIndex`. La celda no imprime nada.

In [6]:
promedio = esolmet_h.groupby(by=esolmet_h.index.strftime("%m-%d %H:%M")).mean()
promedio.index = "2024-" + promedio.index 
promedio.index = pd.to_datetime(promedio.index,format="%Y-%m-%d %H:%M")

## Comprobar

8784 entradas del 1 de enero al 31 de diciembre de 2024 a las 23 h, sin faltantes.

In [7]:
promedio.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8784 entries, 2024-01-01 00:00:00 to 2024-12-31 23:00:00
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   dhi       8784 non-null   float64
 1   dni       8784 non-null   float64
 2   ghi       8784 non-null   float64
 3   p_atm     8784 non-null   float64
 4   rain_acc  8784 non-null   float64
 5   rh        8784 non-null   float64
 6   tdb       8784 non-null   float64
 7   uv        8784 non-null   float64
 8   wd        8784 non-null   float64
 9   ws        8784 non-null   float64
dtypes: float64(10)
memory usage: 754.9 KB


El año promedio completo.

In [8]:
promedio

variable,dhi,dni,ghi,p_atm,rain_acc,rh,tdb,uv,wd,ws
date,,,,,,,,,,
2024-01-01 00:00:00,0.0,0.0,0.0,873.931783,0.000000,53.409167,17.836633,0.000000,295.680175,1.413860
2024-01-01 01:00:00,0.0,0.0,0.0,873.847656,0.006667,56.673389,16.722500,0.000000,313.849133,1.788290
2024-01-01 02:00:00,0.0,0.0,0.0,873.520090,0.000000,58.403583,16.212367,0.000004,305.239542,1.901670
2024-01-01 03:00:00,0.0,0.0,0.0,873.576894,0.000000,60.288917,15.727033,0.000004,309.496996,1.816437
2024-01-01 04:00:00,0.0,0.0,0.0,873.767469,0.000000,62.737944,15.100200,0.000013,274.154650,1.948827
...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,0.0,0.0,0.0,862.110272,0.000333,44.320222,21.350900,0.055713,209.548804,2.418727
2024-12-31 20:00:00,0.0,0.0,0.0,862.007002,0.000000,48.934444,20.198333,0.062567,181.244776,1.550003
2024-12-31 21:00:00,0.0,0.0,0.0,862.532007,1.537333,49.993694,19.806367,0.059379,262.768659,1.402153


## Guardar en `data/003_results/`

`003_results/` es la carpeta para productos: lo que ya no es intermedio sino resultado que
otras libretas consumen. La celda no imprime nada; el resultado es el archivo.

In [9]:
promedio.to_parquet("../data/003_results/anio_promedio-2020-2026.parquet")

Celda vacía que quedó al final de la clase.